In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display  # noqa: F401
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from src.preprocessing import build_preprocessor, prepare_data

warnings.filterwarnings('ignore')
pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

os.makedirs(".doc/notebook_plots/02_Baseline", exist_ok=True)

# PART 2: BASELINE LINEAR REGRESSION MODEL

## Objective
Build a simple LinearRegression model to establish baseline performance and understand feature importance.

### Key Tasks
1. Load cleaned data from Part 1
2. Feature engineering & preprocessing
3. Train-test split
4. Train LinearRegression model
5. Evaluate performance (MSE, RMSE, R2, MAE)
6. Analyze coefficients for feature importance
7. Check for overfitting (train vs test performance)
8. Visualize predictions vs actuals

## Section 1: Load & Prepare Data

We use `prepare_data(strategy='drop')` to remove rows with missing values rather than imputing them, ensuring the baseline model trains on unaltered data so its errors reflect model limitations, not imputation artifacts.

In [2]:
data = prepare_data(strategy='drop')
df_clean = data['df_clean']
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
feature_names = data['feature_names']
preprocessor = data['preprocessor']

split_idx = X_train.shape[0]

display(Markdown(
    f"**Dataset split:** {X_train.shape[0]} train / {X_test.shape[0]} test samples "
    f"| {X_train.shape[1]} features\n\n"
    f"| Set | From | To |\n|---|---|---|\n"
    f"| Train | {df_clean.iloc[:split_idx]['Date'].min().date()} | {df_clean.iloc[:split_idx]['Date'].max().date()} |\n"
    f"| Test | {df_clean.iloc[split_idx:]['Date'].min().date()} | {df_clean.iloc[split_idx:]['Date'].max().date()} |"
))

**Dataset split:** 49 train / 22 test samples | 13 features

| Set | From | To |
|---|---|---|
| Train | 2010-02-05 | 2012-02-03 |
| Test | 2012-02-10 | 2012-10-19 |

## Distribution Shift Analysis

A chronological train/test split can introduce distribution shift: the test period may have different seasonality, trends, or sales levels than the training period. Quantifying this shift before modeling helps distinguish genuine model failure from data drift.

In [3]:
train_split = df_clean.iloc[:split_idx]
test_split = df_clean.iloc[split_idx:]

year_dist = pd.DataFrame({
    'Train': train_split['Year'].value_counts().sort_index(),
    'Test': test_split['Year'].value_counts().sort_index(),
}).fillna(0).astype(int)

month_dist = pd.DataFrame({
    'Train': train_split['Month'].value_counts().sort_index(),
    'Test': test_split['Month'].value_counts().sort_index(),
}).fillna(0).astype(int)

sales_shift_pct = (y_test.mean() - y_train.mean()) / y_train.mean() * 100

display(Markdown("### Year Distribution"))
display(year_dist)

display(Markdown("### Month Distribution"))
display(month_dist)

display(Markdown(
    f"### Weekly Sales Shift\n\n"
    f"| | Mean | Std |\n|---|---|---|\n"
    f"| Train | ${y_train.mean():,.0f} | ${y_train.std():,.0f} |\n"
    f"| Test | ${y_test.mean():,.0f} | ${y_test.std():,.0f} |\n\n"
    f"**Shift: {sales_shift_pct:+.1f}%**"
))

### Year Distribution

,Train,Test
Year,,
2010.0,28,0
2011.0,20,0
2012.0,1,22


### Month Distribution

,Train,Test
Month,,
1.0,1,0
2.0,4,3
3.0,3,5
4.0,3,3
5.0,5,3
6.0,6,2
7.0,8,1
8.0,4,0
9.0,3,1


### Weekly Sales Shift

| | Mean | Std |
|---|---|---|
| Train | $1,268,870 | $709,629 |
| Test | $1,083,133 | $649,723 |

**Shift: -14.6%**

## Section 2: Train LinearRegression Model

Linear regression serves as the simplest interpretable baseline. Its performance sets the floor that more complex models (Ridge, Lasso, tree-based) must beat to justify their added complexity.

In [4]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)

y_train_pred = baseline.predict(X_train)
y_test_pred = baseline.predict(X_test)

display(Markdown(
    f"### LinearRegression Trained\n\n"
    f"| Property | Value |\n|---|---|\n"
    f"| Coefficients | {len(baseline.coef_)} |\n"
    f"| Intercept | ${baseline.intercept_:,.2f} |\n"
    f"| Train predictions | {len(y_train_pred)} |\n"
    f"| Test predictions | {len(y_test_pred)} |"
))

### LinearRegression Trained

| Property | Value |
|---|---|
| Coefficients | 13 |
| Intercept | $442,447.74 |
| Train predictions | 49 |
| Test predictions | 22 |

## Section 3: Model Evaluation

Comparing train and test metrics reveals overfitting: a large gap means the model memorizes training patterns rather than learning generalizable relationships. A negative test R2 signals the model is worse than simply predicting the mean.

In [5]:
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

metrics_df = pd.DataFrame({
    'Train': [f'{train_mse:,.0f}', f'${train_rmse:,.0f}', f'${train_mae:,.0f}', f'{train_r2:.4f}'],
    'Test': [f'{test_mse:,.0f}', f'${test_rmse:,.0f}', f'${test_mae:,.0f}', f'{test_r2:.4f}'],
    'Difference': [
        f'{abs(test_mse - train_mse):,.0f}',
        f'${abs(test_rmse - train_rmse):,.0f}',
        f'${abs(test_mae - train_mae):,.0f}',
        f'{abs(test_r2 - train_r2):.4f}',
    ],
}, index=['MSE', 'RMSE', 'MAE', 'R2'])

display(Markdown("### Model Performance"))
display(metrics_df)

rmse_diff = test_rmse - train_rmse
rmse_diff_pct = (rmse_diff / train_rmse) * 100
r2_diff = train_r2 - test_r2
r2_diff_pct = (r2_diff / train_r2) * 100 if train_r2 > 0 else 0

display(Markdown(
    f"### Overfitting Analysis\n\n"
    f"| Gap | Absolute | Relative |\n|---|---|---|\n"
    f"| RMSE (Test - Train) | ${rmse_diff:,.0f} | {rmse_diff_pct:.1f}% |\n"
    f"| R2 (Train - Test) | {r2_diff:.4f} | {r2_diff_pct:.1f}% |\n\n"
    f"RMSE gap of {rmse_diff_pct:.1f}% indicates substantial overfitting. "
    f"Negative test R2 ({test_r2:.4f}) means the model predicts worse than a simple mean baseline."
))

### Model Performance

,Train,Test,Difference
MSE,"177,277,261,215","1,052,911,740,247","875,634,479,032"
RMSE,"$421,043","$1,026,115","$605,072"
MAE,"$359,202","$923,021","$563,820"
R2,0.6406,-1.6130,2.2536


### Overfitting Analysis

| Gap | Absolute | Relative |
|---|---|---|
| RMSE (Test - Train) | $605,072 | 143.7% |
| R2 (Train - Test) | 2.2536 | 351.8% |

RMSE gap of 143.7% indicates substantial overfitting. Negative test R2 (-1.6130) means the model predicts worse than a simple mean baseline.

### Cross-Validation with TimeSeriesSplit

A single train/test split can be misleading if the cut point happens to be favorable or unfavorable. TimeSeriesSplit provides multiple chronological folds, giving a more robust estimate of out-of-sample performance while respecting the temporal order of the data.

In [6]:
X_full_raw = data['df_clean'].drop(columns=['Weekly_Sales', 'Date', 'Store'])[data['categorical_features'] + data['numeric_features']]
y_full = data['df_clean']['Weekly_Sales']

preprocessor_cv = build_preprocessor(data['categorical_features'], data['numeric_features'])

X_full = preprocessor_cv.fit_transform(X_full_raw)

tscv = TimeSeriesSplit(n_splits=3)
cv_scores = cross_val_score(LinearRegression(), X_full, y_full, cv=tscv, scoring='neg_root_mean_squared_error')

cv_df = pd.DataFrame({
    'Fold': [f'Fold {i+1}' for i in range(len(cv_scores))],
    'RMSE': [f'${-s:,.0f}' for s in cv_scores],
})

display(Markdown("### Cross-Validation (TimeSeriesSplit, 3 folds)"))
display(cv_df.set_index('Fold'))
display(Markdown(
    f"**Mean RMSE:** ${-cv_scores.mean():,.0f} "
    f"(+/- ${cv_scores.std():,.0f})"
))

### Cross-Validation (TimeSeriesSplit, 3 folds)

,RMSE
Fold,
Fold 1,"$1,027,837"
Fold 2,"$659,510"
Fold 3,"$536,705"


**Mean RMSE:** $741,351 (+/- $208,688)

## Section 4: Feature Importance Analysis

Linear regression coefficients directly quantify each feature's contribution to the prediction. Inspecting them reveals which features drive sales and whether any coefficients are unreasonably large, which would indicate multicollinearity or overfitting.

In [7]:
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': baseline.coef_,
    'Abs_Coefficient': np.abs(baseline.coef_),
    'Type': ['categorical' if f.startswith('cat__') else 'numerical' for f in feature_names]
}).sort_values('Abs_Coefficient', ascending=False)

display(Markdown(f"**Model Intercept:** ${baseline.intercept_:,.2f}"))

display(coef_df[['Feature', 'Coefficient', 'Type']].style
    .format({'Coefficient': '${:,.0f}'})
    .set_caption("Feature Coefficients")
    .hide(axis='index'))

display(Markdown(
    "- **Categorical coefficients:** direct impact relative to reference category\n"
    "- **Numerical coefficients:** impact per 1 standard deviation change (features are scaled)"
))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=coef_df['Coefficient'],
    y=coef_df['Feature'],
    orientation='h',
    marker=dict(
        color=coef_df['Coefficient'],
        colorscale='RdBu',
        colorbar=dict(title="Coefficient Value")
    )
))
fig.update_layout(
    title='Feature Coefficients (LinearRegression)',
    xaxis_title='Coefficient Value',
    yaxis_title='Feature',
    height=500,
    showlegend=False
)
fig.write_image(".doc/notebook_plots/02_Baseline/feature_coefficients.png")
fig.show()

**Model Intercept:** $442,447.74

Feature,Coefficient,Type
cat__Store_Grouped_13,"$1,313,123",categorical
cat__Store_Grouped_999,"$984,340",categorical
cat__Store_Grouped_19,"$888,720",categorical
num__Year,"$527,412",numerical
num__Fuel_Price,"$-438,180",numerical
cat__Holiday_Flag_1,"$370,037",categorical
num__CPI,"$-189,955",numerical
cat__Store_Grouped_7,"$179,097",categorical
num__Unemployment,"$132,050",numerical
num__Day,"$81,260",numerical


- **Categorical coefficients:** direct impact relative to reference category
- **Numerical coefficients:** impact per 1 standard deviation change (features are scaled)

### Expected output

![feature_coefficients](.doc/notebook_plots/02_Baseline/feature_coefficients.png)

**Observation:** Store-level categorical features (**Store_Grouped** encoded categories) dominate the coefficient magnitudes, while continuous economic indicators (CPI, Fuel_Price, Unemployment) have comparatively small effects. This suggests that sales variation is primarily driven by which store we are looking at, not by macroeconomic conditions. The large spread in coefficient values also hints at potential multicollinearity among the one-hot encoded store features.

### Note on DayOfWeek

All dates in the Walmart dataset fall on **Friday** (DayOfWeek=4), so this feature has zero variance. The preprocessing pipeline in `src/preprocessing.py` automatically excludes DayOfWeek when its standard deviation is 0 (see `get_feature_lists()`). If it still appears in the feature set, its coefficient will be exactly 0 after StandardScaler normalization, contributing nothing to predictions.

## Section 5: Predictions vs Actuals Visualization

Plotting predicted vs actual values reveals systematic biases: points should cluster around the diagonal line. Deviations from the diagonal expose where the model under- or over-predicts, and scatter width indicates prediction uncertainty.

In [8]:
train_residuals = y_train - y_train_pred
test_residuals = y_test - y_test_pred

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Train Set', 'Test Set'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}]]
)

fig.add_trace(
    go.Scatter(
        x=y_train,
        y=y_train_pred,
        mode='markers',
        marker=dict(color='steelblue', opacity=0.6, size=6),
        name='Train predictions'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=y_test,
        y=y_test_pred,
        mode='markers',
        marker=dict(color='coral', opacity=0.6, size=6),
        name='Test predictions'
    ),
    row=1, col=2
)

min_val = min(y_train.min(), y_test.min())
max_val = max(y_train.max(), y_test.max())
perfect_line = np.array([min_val, max_val])

fig.add_trace(
    go.Scatter(
        x=perfect_line,
        y=perfect_line,
        mode='lines',
        line=dict(color='red', width=2, dash='dash'),
        name='Perfect predictions'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=perfect_line,
        y=perfect_line,
        mode='lines',
        line=dict(color='red', width=2, dash='dash'),
        name='Perfect predictions',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_xaxes(title_text='Actual Sales ($)', row=1, col=1)
fig.update_yaxes(title_text='Predicted Sales ($)', row=1, col=1)
fig.update_xaxes(title_text='Actual Sales ($)', row=1, col=2)
fig.update_yaxes(title_text='Predicted Sales ($)', row=1, col=2)

fig.update_layout(
    title='Predictions vs Actual Values',
    height=450,
    hovermode='closest'
)
fig.write_image(".doc/notebook_plots/02_Baseline/predictions_vs_actuals.png")
fig.show()

### Expected output

![predictions_vs_actuals](.doc/notebook_plots/02_Baseline/predictions_vs_actuals.png)

**Observation:** Train predictions cluster around the diagonal with moderate scatter. Test predictions deviate significantly, with the model tending to overshoot (predicted > actual), reflecting the distribution shift between train and test periods.

## Section 6: Residuals Analysis

Residual plots diagnose whether the model's errors are random (good) or structured (bad). Patterns such as funneling, curvature, or heavy tails indicate violated assumptions and guide what improvements are needed.

In [9]:
residuals_df = pd.DataFrame({
    'Train': train_residuals.describe(),
    'Test': test_residuals.describe(),
})

display(Markdown("### Residuals Summary"))
display(residuals_df.drop('count').style.format('${:,.0f}').set_caption("Residuals Statistics"))

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Train Residuals', 'Test Residuals', 'Train Residuals Distribution', 'Test Residuals Distribution'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}], [{"type": "histogram"}, {"type": "histogram"}]]
)

fig.add_trace(
    go.Scatter(
        x=y_train_pred,
        y=train_residuals,
        mode='markers',
        marker=dict(color='steelblue', opacity=0.6, size=6),
        name='Train'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=y_test_pred,
        y=test_residuals,
        mode='markers',
        marker=dict(color='coral', opacity=0.6, size=6),
        name='Test'
    ),
    row=1, col=2
)

fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)

fig.add_trace(
    go.Histogram(
        x=train_residuals,
        nbinsx=20,
        marker_color='steelblue',
        opacity=0.7,
        name='Train hist'
    ),
    row=2, col=1
)

fig.add_trace(
    go.Histogram(
        x=test_residuals,
        nbinsx=20,
        marker_color='coral',
        opacity=0.7,
        name='Test hist'
    ),
    row=2, col=2
)

fig.update_xaxes(title_text='Predicted Sales ($)', row=1, col=1)
fig.update_yaxes(title_text='Residual ($)', row=1, col=1)
fig.update_xaxes(title_text='Predicted Sales ($)', row=1, col=2)
fig.update_yaxes(title_text='Residual ($)', row=1, col=2)
fig.update_xaxes(title_text='Residual ($)', row=2, col=1)
fig.update_yaxes(title_text='Frequency', row=2, col=1)
fig.update_xaxes(title_text='Residual ($)', row=2, col=2)
fig.update_yaxes(title_text='Frequency', row=2, col=2)

fig.update_layout(
    title='Residuals Analysis (LinearRegression)',
    height=700,
    showlegend=False
)
fig.write_image(".doc/notebook_plots/02_Baseline/residuals_analysis.png")
fig.show()

### Residuals Summary

,Train,Test
mean,$0,"$-923,021"
std,"$425,406","$458,816"
min,"$-742,075","$-1,715,132"
25%,"$-367,401","$-1,327,051"
50%,"$-56,351","$-815,834"
75%,"$354,433","$-593,519"
max,"$853,196","$-239,807"


### Expected output

![residuals_analysis](.doc/notebook_plots/02_Baseline/residuals_analysis.png)

**Observation:** Funnel-shaped residual scatter (heteroscedasticity) confirms variance increases with predicted values. Test residuals are predominantly negative, indicating the model systematically overestimates sales in the test period. The test residual distribution is also left-skewed, with a longer tail extending toward large negative errors. Train residuals are approximately symmetric around zero. These patterns suggest a log-transform of the target or a model that handles non-linear relationships would improve performance.

## Summary

Consolidating all findings into a single view to establish the baseline performance reference that subsequent models must improve upon.

In [10]:
cv_mean_rmse = -cv_scores.mean()
cv_std_rmse = cv_scores.std()
total_samples = X_train.shape[0] + X_test.shape[0]

display(Markdown(
    f"### Baseline Model Summary\n\n"
    f"**Dataset:** {total_samples} samples ({X_train.shape[0]} train / {X_test.shape[0]} test), "
    f"{X_train.shape[1]} features\n\n"
    f"| Metric | Train | Test |\n|---|---|---|\n"
    f"| RMSE | ${train_rmse:,.0f} | ${test_rmse:,.0f} |\n"
    f"| MAE | ${train_mae:,.0f} | ${test_mae:,.0f} |\n"
    f"| R2 | {train_r2:.4f} | {test_r2:.4f} |\n\n"
    f"**Overfitting:** RMSE gap of {rmse_diff_pct:.1f}% between train and test. "
    f"Test R2 = {test_r2:.4f} (negative R2 means predictions are worse than the mean baseline).\n\n"
    f"**Cross-validation:** Mean RMSE = ${cv_mean_rmse:,.0f} (+/- ${cv_std_rmse:,.0f}) "
    f"over 3 TimeSeriesSplit folds.\n\n"
    f"**Conclusion:** The linear regression baseline overfits heavily and fails to generalize "
    f"under chronological split. Regularization (Ridge/Lasso) is needed in Part 3."
))

### Baseline Model Summary

**Dataset:** 71 samples (49 train / 22 test), 13 features

| Metric | Train | Test |
|---|---|---|
| RMSE | $421,043 | $1,026,115 |
| MAE | $359,202 | $923,021 |
| R2 | 0.6406 | -1.6130 |

**Overfitting:** RMSE gap of 143.7% between train and test. Test R2 = -1.6130 (negative R2 means predictions are worse than the mean baseline).

**Cross-validation:** Mean RMSE = $741,351 (+/- $208,688) over 3 TimeSeriesSplit folds.

**Conclusion:** The linear regression baseline overfits heavily and fails to generalize under chronological split. Regularization (Ridge/Lasso) is needed in Part 3.